# Cart-Pole + Vuepyter

A cart moves along a horizontal track, and a rigid rod is pinned to the cart. The upright rod is unstable, so the goal is to keep `theta = 0` while sending a normalized speed command in `[-1, 1]`.

The live figure on the left advances in real time. The notebook defines both the physics model and a tiny feedback controller.

In [1]:
from __future__ import annotations

import math


def clamp(value: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(value)))


class CartPoleSim:
    """A compact inverted-pendulum model driven by a desired cart speed."""

    def __init__(
        self,
        *,
        x: float = 0.0,
        x_dot: float = 0.0,
        theta: float = 0.12,
        theta_dot: float = 0.0,
        dt: float = 1 / 120,
    ):
        self.dt = dt
        self.max_speed = 2.0
        self.speed_tau = 0.04
        self.pole_length = 0.8
        self.gravity = 9.81
        self.angular_damping = 0.1
        self.track_limit = 2.4
        self.time = 0.0
        self.command = 0.0
        self.terminated = False
        self.reset(x=x, x_dot=x_dot, theta=theta, theta_dot=theta_dot)

    @property
    def state(self) -> dict[str, float | bool]:
        return {
            "time": self.time,
            "x": self.x,
            "x_dot": self.x_dot,
            "theta": self.theta,
            "theta_dot": self.theta_dot,
            "command": self.command,
            "pole_length": self.pole_length,
            "track_limit": self.track_limit,
            "terminated": self.terminated,
        }

    def reset(
        self,
        *,
        x: float = 0.0,
        x_dot: float = 0.0,
        theta: float = 0.12,
        theta_dot: float = 0.0,
    ) -> dict[str, float | bool]:
        self.x = float(x)
        self.x_dot = float(x_dot)
        self.theta = float(theta)
        self.theta_dot = float(theta_dot)
        self.time = 0.0
        self.command = 0.0
        self.terminated = False
        return self.state

    def step(self, speed_cmd: float, dt: float | None = None) -> dict[str, float | bool]:
        if dt is not None:
            self.dt = clamp(dt, 1 / 240, 1 / 30)

        if self.terminated:
            return self.state

        self.command = clamp(speed_cmd, -1.0, 1.0)
        desired_speed = self.command * self.max_speed
        x_ddot = (desired_speed - self.x_dot) / self.speed_tau

        self.x_dot += x_ddot * self.dt
        self.x += self.x_dot * self.dt
        if self.x < -self.track_limit:
            self.x = -self.track_limit
            self.x_dot = max(0.0, self.x_dot)
        elif self.x > self.track_limit:
            self.x = self.track_limit
            self.x_dot = min(0.0, self.x_dot)

        theta_ddot = (
            (self.gravity / self.pole_length) * math.sin(self.theta)
            - (x_ddot / self.pole_length) * math.cos(self.theta)
            - self.angular_damping * self.theta_dot
        )
        self.theta_dot += theta_ddot * self.dt
        self.theta += self.theta_dot * self.dt
        self.theta = (self.theta + math.pi) % (2 * math.pi) - math.pi
        self.time += self.dt

        if abs(self.theta) > math.pi / 2:
            self.terminated = True

        return self.state


The state is `(x, x_dot, theta, theta_dot)`, where `theta = 0` means perfectly upright. The control input is a desired cart speed, and the cart reacts with a short time constant so that the pole feels the base acceleration.

In [ ]:
sim = CartPoleSim(theta=0.14)
manual_command = 0.0
auto_balance = True

sim.state

A tiny stabilizer can already do the job. The controller below reads the angle, angular rate, cart position, and cart speed, then blends them into one bounded speed command. The gains were tuned for this simplified model so the rod is caught quickly and the cart drifts back toward the middle.

In [ ]:
def controller(state: dict[str, float | bool]) -> float:
    theta = float(state["theta"])
    theta_dot = float(state["theta_dot"])
    x = float(state["x"])
    x_dot = float(state["x_dot"])

    command = 9.06 * theta + 2.77 * theta_dot + 0.72 * x + 1.87 * x_dot
    return clamp(command, -1.0, 1.0)


def choose_command(state: dict[str, float | bool]) -> float:
    return controller(state) if auto_balance else manual_command


choose_command(sim.state)

Try editing and re-running the next cell. For a manual experiment, set `auto_balance = False` and choose a `manual_command` in `[-1, 1]`. To watch the simple algorithm recover, reset to a tilted initial condition and keep `auto_balance = True`.

In [ ]:
# Manual experiment:
# auto_balance = False
# manual_command = -0.25

# Automatic recovery:
sim.reset(theta=0.14, theta_dot=0.0, x=0.0, x_dot=0.0)
auto_balance = True

sim.state